# Notebook 1: Data Exploration & EDA
## When Do Models Fail? Discovering Hidden Failure Patterns in Spatio-Temporal Ride Demand Prediction

This notebook covers:
1. Loading and exploring NYC TLC Yellow Taxi data (2019)
2. Aggregating ride counts by zone and 30-minute time windows
3. Exploratory Data Analysis with visualizations
4. Feature engineering overview

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_ingestion import load_config, download_taxi_data, download_zone_lookup, aggregate_demand
from src.feature_engineering import engineer_features

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

config = load_config('../configs/config.yaml')
print('Configuration loaded.')

## 1. Data Download
Download NYC TLC Yellow Taxi trip records for all 12 months of 2019.

In [ ]:
# Download raw parquet files (skip if already downloaded)
download_taxi_data(config)

# Download zone lookup table
zone_lookup = download_zone_lookup(config)
print(f'\nZone lookup: {len(zone_lookup)} zones')
zone_lookup.head(10)

## 2. Demand Aggregation
Aggregate trip counts into 30-minute windows per taxi zone.

In [ ]:
demand_df = aggregate_demand(config)

print(f'Shape: {demand_df.shape}')
print(f'Time range: {demand_df["time_window"].min()} to {demand_df["time_window"].max()}')
print(f'Zones: {demand_df["zone_id"].nunique()}')
print(f'Total trips: {demand_df["trip_count"].sum():,}')
print(f'\nZero-demand entries: {(demand_df["trip_count"] == 0).sum():,} ({(demand_df["trip_count"] == 0).mean()*100:.1f}%)')

demand_df.head()

In [ ]:
# Basic statistics
print('Trip count statistics:')
demand_df['trip_count'].describe()

## 3. Exploratory Data Analysis

In [ ]:
# Add temporal columns for EDA
demand_df['hour'] = pd.to_datetime(demand_df['time_window']).dt.hour
demand_df['day_of_week'] = pd.to_datetime(demand_df['time_window']).dt.dayofweek
demand_df['month'] = pd.to_datetime(demand_df['time_window']).dt.month

In [ ]:
# Plot 1: Average demand by hour of day
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

hourly = demand_df.groupby('hour')['trip_count'].mean()
axes[0].bar(hourly.index, hourly.values, color='#1a73e8', alpha=0.85)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Avg Trip Count per Zone')
axes[0].set_title('Average Demand by Hour')
axes[0].set_xticks(range(0, 24, 2))

# Plot 2: Demand by day of week
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily = demand_df.groupby('day_of_week')['trip_count'].mean()
axes[1].bar(range(7), daily.values, color='#34a853', alpha=0.85)
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(dow_names)
axes[1].set_ylabel('Avg Trip Count per Zone')
axes[1].set_title('Average Demand by Day of Week')

plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Demand heatmap (Hour x Day of Week)
pivot = demand_df.groupby(['day_of_week', 'hour'])['trip_count'].mean().reset_index()
heatmap_data = pivot.pivot(index='day_of_week', columns='hour', values='trip_count')

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=False, ax=ax,
            xticklabels=range(24), yticklabels=dow_names,
            cbar_kws={'label': 'Avg Trip Count'})
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')
ax.set_title('Average Ride Demand Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: Demand distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(demand_df['trip_count'], bins=100, color='#1a73e8', alpha=0.7, edgecolor='white')
axes[0].set_xlabel('Trip Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Demand Distribution (All)')
axes[0].set_yscale('log')

nonzero = demand_df[demand_df['trip_count'] > 0]['trip_count']
axes[1].hist(nonzero, bins=100, color='#34a853', alpha=0.7, edgecolor='white')
axes[1].set_xlabel('Trip Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Demand Distribution (Non-Zero)')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print(f'Percentiles of non-zero demand:')
for p in [25, 50, 75, 90, 95, 99]:
    print(f'  P{p}: {np.percentile(nonzero, p):.0f}')

In [ ]:
# Plot 5: Top zones by demand
zone_demand = demand_df.groupby('zone_id')['trip_count'].agg(['mean', 'std', 'max']).sort_values('mean', ascending=False)

if zone_lookup is not None:
    zone_demand = zone_demand.merge(zone_lookup[['LocationID', 'Zone', 'Borough']], 
                                     left_index=True, right_on='LocationID', how='left')

print('Top 15 zones by average demand:')
zone_demand.head(15)

In [ ]:
# Plot 6: Monthly total demand trend
monthly = demand_df.groupby('month')['trip_count'].sum()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(monthly.index, monthly.values / 1e6, color='#1a73e8', alpha=0.85)
ax.set_xlabel('Month')
ax.set_ylabel('Total Trips (Millions)')
ax.set_title('Monthly Total Taxi Trips (2019)', fontsize=14, fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                     'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.tight_layout()
plt.show()

## 4. Feature Engineering Preview

In [ ]:
features_df = engineer_features(config)

print(f'Engineered features shape: {features_df.shape}')
print(f'\nAll columns:')
for col in features_df.columns:
    print(f'  {col}: {features_df[col].dtype}')

features_df.head()

In [ ]:
# Feature correlation with target
target = config['features']['target']
exclude = ['time_window', 'zone_id', target]
feat_cols = [c for c in features_df.columns if c not in exclude]

correlations = features_df[feat_cols + [target]].corr()[target].drop(target).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#34a853' if v > 0 else '#d93025' for v in correlations]
ax.barh(range(len(correlations)), correlations.values, color=colors, alpha=0.8)
ax.set_yticks(range(len(correlations)))
ax.set_yticklabels(correlations.index)
ax.invert_yaxis()
ax.set_xlabel('Correlation with Trip Count')
ax.set_title('Feature Correlation with Target', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## Key EDA Takeaways

Document your observations here:
- **Demand is highly skewed**: Most zone-window combinations have low demand, while a few hotspots dominate
- **Clear temporal patterns**: Demand peaks during rush hours and varies by day of week
- **Spatial heterogeneity**: Manhattan zones have dramatically higher demand than outer boroughs
- **Zero inflation**: A large fraction of zone-window pairs have zero demand (low-activity zones)